In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, GRU, RNN, Dense, Embedding, Attention, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [2]:
# Define the model parameters
vocab_size = 10000
embedding_dim = 256
max_seq_len = 30
latent_dim = 512


In [3]:
# Encoder - (RNN, LSTM, or GRU)

def build_encoder(input_dim, latent_dim, seq_len):
  encoder_inputs = Input(shape=(seq_len,))
  x= Embedding(input_dim=input_dim, output_dim=embedding_dim)(encoder_inputs)
  x = LSTM(latent_dim, return_state=True)(x)
  encoder_outputs, state_h, state_c = x

  return encoder_inputs, encoder_outputs, state_h, state_c

In [4]:
# Decoder with Attention Mechanism

def build_decoder(input_dim, latent_dim, seq_len, encoder_outputs, state_h, state_c):
  decoder_inputs = Input(shape=(seq_len,))
  x = Embedding(input_dim=input_dim, output_dim=embedding_dim)(decoder_inputs)

  # Attention Mechanism

  attention = Attention(use_scale = True)([x, encoder_outputs])
  decoder_concat = Concatenate(axis=-1)([x, attention])

  # decoder LSTM
  x = LSTM(latent_dim, return_sequences=True)(decoder_concat, initial_state=[state_h, state_c])
  decoder_outputs = Dense(input_dim, activation='softmax')(x)

  return decoder_inputs, decoder_outputs

In [5]:
def build_seq2seq_model(input_dim, latent_dim, seq_len):
  encoder_inputs, encoder_outputs, state_h, state_c = build_encoder(input_dim, latent_dim, seq_len)
  decoder_inputs, decoder_outputs = build_decoder(input_dim, latent_dim, seq_len, encoder_outputs, state_h, state_c)

  model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
  model.compile(optimizer=Adam(), loss='cross_categorical_crossentropy', metrics=['accuracy'])

  return model

In [6]:
model = build_seq2seq_model(vocab_size, latent_dim, max_seq_len)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 30, 256)   │  2,560,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 30, 256)   │  2,560,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 512),     │  1,574,912 │ embedding[0][0]   │
│                     │ (None, 512),      │            │                   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 30, 512)   │          1 │ embedding_1[0][0… │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 30, 768)   │          0 │ embedding_1[0][0… │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 30, 512)   │  2,623,488 │ concatenate[0][0… │
│                     │                   │            │ lstm[0][1],       │
│                     │                   │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 30, 10000) │  5,130,000 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,448,401 (55.12 MB)

 Trainable params: 14,448,401 (55.12 MB)

 Non-trainable params: 0 (0.00 B)